In [2]:
%pip install optuna --user

# IMPORTS
import pandas as pd
import numpy as np
import pickle
import os
os.environ['PYTHONWARNINGS'] = 'ignore'

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    RepeatedStratifiedKFold,
    cross_validate, cross_val_score
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix,
    classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

import warnings
warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)

import optuna

from uncertainty_transformer import UncertaintyTransformer

# reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
# CONSTANTS

EXCEL_PATH = "copy_Miyokardit_08.12.2025.xlsx"
LABEL_COL = "GRUP"
NAN_COL_THRESHOLD = 0.20  # drop columns with more than 20% NaN
from uncertainty_utils import EPS
from uncertainty_utils import N_BINS

In [4]:
# DATA LOADING AND PREPROCESSING

df_raw = pd.read_excel(EXCEL_PATH)

label_series = df_raw[LABEL_COL]

df_part1 = df_raw.iloc[:, 6:37]   # G..AK
df_part2 = df_raw.iloc[:, 41:61]  # AP..BI
df_part3 = df_raw.iloc[:, 63:64]  # BL
df = pd.concat([df_part1, df_part2, df_part3], axis=1)

# manually exclude columns not present in the finetuned model
COLS_TO_EXCLUDE = [
    'HIPERTIROIDI',
    'HIPOTIROIDI',
    'Mental Health History',
    'PAH',
    'ROMATOLOJIK_HASTALIK',
    'Socioeconomic Status',
]
df = df.drop(columns=[c for c in COLS_TO_EXCLUDE if c in df.columns])

# hidden NaN filtering
df = df.replace([" ", "", "-", "--", "nan", "NaN", "None",
                 "#VALUE!", "#N/A", "#REF!", "#DIV/0!", "#NUM!", "#NAME?", "#NULL!"], pd.NA)
df = df.apply(lambda col: col.replace(r'^\s*$', pd.NA, regex=True))

# drop datetime cols if any
datetime_cols = df.select_dtypes(include=["datetime"]).columns
df = df.drop(columns=datetime_cols)

# numeric coercion
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# reattach label
df[LABEL_COL] = label_series

print("Final shape:", df.shape)
print("Class distribution:")
print(df[LABEL_COL].value_counts(dropna=False).sort_index())

Final shape: (184, 47)
Class distribution:
GRUP
1     83
2    101
Name: count, dtype: int64


In [5]:
# load split indices
with open('split_indices.pkl', 'rb') as f:
    split_indices = pickle.load(f)

train_idx = pd.Index(split_indices['train_idx'])
test_idx = pd.Index(split_indices['test_idx'])

# Check 1: All train indices exist in df
assert set(train_idx).issubset(df.index), (
    f"FROZEN SPLIT INTEGRITY ERROR: Some train indices are missing from df. "
    f"Missing: {set(train_idx) - set(df.index)}. "
    f"This suggests preprocessing changed or Excel file changed."
)

# Check 2: All test indices exist in df
assert set(test_idx).issubset(df.index), (
    f"FROZEN SPLIT INTEGRITY ERROR: Some test indices are missing from df. "
    f"Missing: {set(test_idx) - set(df.index)}. "
    f"This suggests preprocessing changed or Excel file changed."
)

# Check 3: Train and test indices do not overlap
assert len(set(train_idx) & set(test_idx)) == 0, (
    f"FROZEN SPLIT INTEGRITY ERROR: Train and test indices overlap. "
    f"Overlapping indices: {set(train_idx) & set(test_idx)}. "
    f"This suggests corrupted split_indices.pkl file."
)

print("✓ Frozen split integrity checks passed")
print(f"  Train indices: {len(train_idx)} (all present in df)")
print(f"  Test indices: {len(test_idx)} (all present in df)")
print(f"  No overlap between train and test indices")

# materialize TRAIN data
df_train = df.loc[train_idx].copy()

print(f"\nTrain set (before detailed preprocessing): {len(df_train)} samples")
print(f"Test set (indices only, not materialized): {len(test_idx)} samples")
print(f"\nSplit loaded from: split_indices.pkl")
print(f"  random_state: {split_indices['random_state']}")
print(f"  test_size: {split_indices['test_size']}")

✓ Frozen split integrity checks passed
  Train indices: 147 (all present in df)
  Test indices: 37 (all present in df)
  No overlap between train and test indices

Train set (before detailed preprocessing): 147 samples
Test set (indices only, not materialized): 37 samples

Split loaded from: split_indices.pkl
  random_state: 42
  test_size: 0.2


In [6]:
# extract features and labels from raw training data

# get all numeric features 
features = [c for c in df_train.columns if c != LABEL_COL and df_train[c].dtype in ['int64', 'float64']]

# extract raw data 
X_train_raw = df_train[features].copy()
y_train = df_train[LABEL_COL].values

print(f"Raw training data shape: {X_train_raw.shape}")
print(f"Number of features: {len(features)}")
print(f"\nClass distribution (raw, before CV preprocessing):")
class_counts = pd.Series(y_train).value_counts().sort_index()
for label, count in class_counts.items():
    print(f"  Class {label}: {count} patients")

Raw training data shape: (147, 46)
Number of features: 46

Class distribution (raw, before CV preprocessing):
  Class 1: 66 patients
  Class 2: 81 patients


In [7]:
# uncertainty transform uses class conditional statistics
# so it must be fitted inside each CV fold, therefore we should not precompute X_train_scaled
# we also avoid reusing the same transformer instance

def make_pipeline(clf, feature_names):
    """
    Build a pipeline with UncertaintyTransformer using fold-specific feature names.
    
    Parameters
    ----------
    clf : classifier
        The classifier to use in the pipeline.
    feature_names : list of str
        Feature names for this fold (must match X columns exactly).
        
    Notes
    -----
    Since preprocessing eliminates all NaN rows inside each fold, the transformer
    will raise an error if any unexpected NaNs appear.
    is used to catch any unexpected NaNs that might appear.
    """
    return Pipeline([
        (
            "uncertainty",
            UncertaintyTransformer(
                feature_names=feature_names,
                class_labels=None,  # infer from y, enforced to be binary by previous check
                n_bins=N_BINS,
                eps=EPS,
            ),
        ),
        ("scaler", StandardScaler()),
        ("clf", clf),
    ])

print("Pipeline is ready with NaN handling.")

Pipeline is ready with NaN handling.


## INITIAL MODEL SELECTION

In [8]:
initial_models = [
    # LR
    # as it is binary classification, we do not need to specify multi_class -> ovr = multinomial
    LogisticRegression(max_iter=1000, solver='lbfgs', random_state=RANDOM_STATE),
    LogisticRegression(max_iter=500, solver='saga', penalty='l2', C=1.0, random_state=RANDOM_STATE),
    LogisticRegression(max_iter=2000, solver='lbfgs', C=0.5, random_state=RANDOM_STATE),
    LogisticRegression(max_iter=1000, solver='saga', penalty='elasticnet', l1_ratio=0.5, random_state=RANDOM_STATE),
    LogisticRegression(max_iter=1500, solver='liblinear', penalty='l2', C=10.0, random_state=RANDOM_STATE),
    
    # RF
    RandomForestClassifier(n_estimators=100, max_depth=None, max_features='sqrt', random_state=RANDOM_STATE),
    RandomForestClassifier(n_estimators=200, max_depth=50, max_features='sqrt', min_samples_split=4, random_state=RANDOM_STATE),
    RandomForestClassifier(n_estimators=300, max_depth=20, max_features='log2', min_samples_leaf=3, random_state=RANDOM_STATE),
    RandomForestClassifier(n_estimators=150, max_depth=30, max_features=0.8, bootstrap=False, random_state=RANDOM_STATE),
    RandomForestClassifier(n_estimators=500, max_depth=None, min_samples_leaf=5, random_state=RANDOM_STATE),
    
    # SVM
    SVC(probability=True, kernel='rbf', C=1.0, gamma='scale', random_state=RANDOM_STATE),
    SVC(probability=True, kernel='poly', C=0.5, gamma='auto', degree=3, random_state=RANDOM_STATE),
    SVC(probability=True, kernel='linear', C=1.0, random_state=RANDOM_STATE),
    SVC(probability=True, kernel='rbf', C=10.0, gamma=0.1, random_state=RANDOM_STATE),
    SVC(probability=True, kernel='sigmoid', C=0.1, gamma='scale', random_state=RANDOM_STATE),
    
    # KNN
    KNeighborsClassifier(n_neighbors=5, weights='uniform', p=2),
    KNeighborsClassifier(n_neighbors=3, weights='distance', p=1),
    KNeighborsClassifier(n_neighbors=10, weights='distance', p=2),
    KNeighborsClassifier(n_neighbors=7, weights='uniform', p=1),
    KNeighborsClassifier(n_neighbors=15, weights='distance', p=2),
]

print(f"Defined {len(initial_models)} initial models for screening")

Defined 20 initial models for screening


In [9]:
# NESTED CROSS VALIDATION
# outer CV -> unbiased evaluation
# inner CV -> hyperparameter tuning

outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
inner_cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=RANDOM_STATE + 1)  # repeated CV for more robust tuning

# objective functions, only for the best performing models
def make_objective_lr(X_inner, y_inner, inner_cv_split, feature_names):
    def objective_lr(trial):
        penalty = trial.suggest_categorical("penalty", ["l2", "l1", "elasticnet"])
        solver = trial.suggest_categorical("solver", ["lbfgs", "liblinear", "saga"])
        class_weight = trial.suggest_categorical("class_weight", [None, "balanced"])
        
        if penalty == "l1" and solver not in ["liblinear", "saga"]:
            raise optuna.exceptions.TrialPruned()
        if penalty == "elasticnet" and solver != "saga":
            raise optuna.exceptions.TrialPruned()
        
        C = trial.suggest_float("C", 1e-5, 1e3, log=True)
        l1_ratio = trial.suggest_float("l1_ratio", 0.0, 1.0) if penalty == "elasticnet" else None
        
        lr = LogisticRegression(
            penalty=penalty, solver=solver, C=C, l1_ratio=l1_ratio,
            class_weight=class_weight, max_iter=2000, random_state=RANDOM_STATE
        )
        pipe = make_pipeline(lr, feature_names)
        scores = cross_val_score(pipe, X_inner, y_inner, scoring='recall_macro', cv=inner_cv_split, n_jobs=-1)
        return scores.mean()
    return objective_lr

def make_objective_rf(X_inner, y_inner, inner_cv_split, feature_names):
    def objective_rf(trial):
        n_estimators = trial.suggest_int("n_estimators", 100, 1000, step=50)
        max_depth = trial.suggest_int("max_depth", 5, 50)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)
        max_features = trial.suggest_categorical("max_features", ["sqrt", "log2", None])
        class_weight = trial.suggest_categorical("class_weight", [None, "balanced"])
        
        rf = RandomForestClassifier(
            n_estimators=n_estimators, max_depth=max_depth,
            min_samples_split=min_samples_split, min_samples_leaf=min_samples_leaf,
            max_features=max_features, class_weight=class_weight,
            random_state=RANDOM_STATE, n_jobs=-1
        )
        pipe = make_pipeline(rf, feature_names)
        scores = cross_val_score(pipe, X_inner, y_inner, scoring='recall_macro', cv=inner_cv_split, n_jobs=-1)
        return scores.mean()
    return objective_rf

def make_objective_svc(X_inner, y_inner, inner_cv_split, feature_names):
    def objective_svc(trial):
        kernel = trial.suggest_categorical("kernel", ["rbf", "linear", "poly", "sigmoid"])
        C = trial.suggest_float("C", 1e-3, 1e3, log=True)
        gamma = trial.suggest_categorical("gamma", ["scale", "auto"]) if kernel != "linear" else "scale"
        degree = trial.suggest_int("degree", 2, 5) if kernel == "poly" else 3

        svc = SVC(
            kernel=kernel, C=C, gamma=gamma, degree=degree,
            probability=True, random_state=RANDOM_STATE
        )
        pipe = make_pipeline(svc, feature_names)
        scores = cross_val_score(pipe, X_inner, y_inner, scoring='recall_macro', cv=inner_cv_split, n_jobs=-1)
        return scores.mean()
    return objective_svc

def make_objective_knn(X_inner, y_inner, inner_cv_split, feature_names):
    def objective_knn(trial):
        n_neighbors = trial.suggest_int("n_neighbors", 3, 25)
        weights = trial.suggest_categorical("weights", ["uniform", "distance"])
        p = trial.suggest_int("p", 1, 2)

        knn = KNeighborsClassifier(n_neighbors=n_neighbors, weights=weights, p=p)
        pipe = make_pipeline(knn, feature_names)
        scores = cross_val_score(pipe, X_inner, y_inner, scoring='recall_macro', cv=inner_cv_split, n_jobs=-1)
        return scores.mean()
    return objective_knn

objective_map = {
    'LogisticRegression': make_objective_lr,
    'RandomForestClassifier': make_objective_rf,
    'SVC': make_objective_svc,
    'KNeighborsClassifier': make_objective_knn,
}

print("\n" + "=" * 80)
print("NESTED CROSS-VALIDATION")
print("=" * 80)
print(f"Outer CV: {outer_cv.n_splits}-fold StratifiedKFold (evaluation)")
inner_n_splits = inner_cv.get_n_splits() // inner_cv.n_repeats  # splits per repeat
print(f"Inner CV: {inner_n_splits}-fold StratifiedKFold × {inner_cv.n_repeats} repeats (tuning)")
print(f"Screening CV: 5-fold StratifiedKFold (model screening)")
print(f"\nEvaluating all {len(initial_models)} models, selecting top 10 per outer fold...\n")

# store results, one winner per outer fold 
outer_selected_scores = [] 

# also store results per model for diagnostics, not needed for now
model_win_counts = {} 

# track best candidate per model family across all folds
# which combination of these models, when averaged, gives the best validation performance
family_candidates = {}

# screening CV for model selection, used inside each outer fold
screening_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE + 2)
screening_scoring = {'recall': 'recall_macro'}

for outer_fold, (outer_train_idx, outer_val_idx) in enumerate(outer_cv.split(X_train_raw, y_train), 1):
    print(f"\n{'='*80}")
    print(f"OUTER FOLD {outer_fold}/{outer_cv.n_splits}")
    print(f"{'='*80}")
    
    X_outer_train = X_train_raw.iloc[outer_train_idx].copy()
    y_outer_train = y_train[outer_train_idx].copy()
    X_outer_val = X_train_raw.iloc[outer_val_idx].copy()
    y_outer_val = y_train[outer_val_idx].copy()
    
    # ========================================================================
    # PREPROCESSING INSIDE OUTER FOLD
    # ========================================================================

    # step 1: drop columns with >NAN_COL_THRESHOLD NaN from outer_train, apply same to val
    nan_pct = X_outer_train.isna().mean()
    cols_to_drop = nan_pct[nan_pct > NAN_COL_THRESHOLD].index.tolist()

    if cols_to_drop:
        X_outer_train = X_outer_train.drop(columns=cols_to_drop)
        X_outer_val = X_outer_val.drop(columns=cols_to_drop, errors='ignore')

    # step 2: drop rows with any remaining NaNs from outer_train
    initial_rows = len(X_outer_train)
    mask_train = ~X_outer_train.isna().any(axis=1)
    X_outer_train = X_outer_train[mask_train]
    y_outer_train = y_outer_train[mask_train]
    rows_dropped = initial_rows - len(X_outer_train)

    # step 3: define fold_features from actual columns
    fold_features = list(X_outer_train.columns)

    # step 4: ensure validation has same columns in same order
    X_outer_val = X_outer_val[fold_features]

    # step 5: drop rows with NaNs from validation set
    mask_val = ~X_outer_val.isna().any(axis=1)
    X_outer_val = X_outer_val[mask_val]
    y_outer_val = y_outer_val[mask_val]
    
    # step 6: two class guard
    if pd.Series(y_outer_train).nunique() != 2:
        print(f"  Skipping fold {outer_fold}: not enough classes after filtering (got {pd.Series(y_outer_train).nunique()} classes)")
        continue

    if pd.Series(y_outer_val).nunique() < 2:
        print(f"  Warning: outer_val fold {outer_fold} has only 1 class after NaN filtering. Fold skipped.")
        continue
    
    if outer_fold == 1:
        print(f"  Preprocessing (fold {outer_fold}):")
        print(f"    Dropped {len(cols_to_drop)} columns with >{int(NAN_COL_THRESHOLD*100)}% NaN: {cols_to_drop if len(cols_to_drop) <= 5 else cols_to_drop[:5] + ['...']}")
        print(f"    Dropped {rows_dropped} rows with remaining NaNs from outer_train")
        print(f"    Final outer_train shape: {X_outer_train.shape}")
        print(f"    Final outer_val shape: {X_outer_val.shape}")
    
    # ========================================================================
    # STEP 1: SCREEN ALL MODELS
    # ========================================================================
    print(f"  Screening all {len(initial_models)} models on outer_train...")
    fold_screening_results = []
    for model_idx, model in enumerate(initial_models):
        model_name = f"{type(model).__name__}_{model_idx+1}"
        pipe = make_pipeline(model, fold_features)
        cv_results = cross_validate(
            pipe, X_outer_train, y_outer_train,
            cv=screening_cv, scoring=screening_scoring,
            return_train_score=False, n_jobs=-1
        )
        mean_recall = cv_results['test_recall'].mean()
        fold_screening_results.append({
            'model_idx': model_idx,
            'model_name': model_name,
            'model': model,
            'mean_recall': mean_recall,
        })
    
    # select top 10 models for this fold
    fold_screening_df = pd.DataFrame(fold_screening_results)
    fold_screening_df = fold_screening_df.sort_values(by='mean_recall', ascending=False)
    fold_top_10_indices = fold_screening_df.head(10)['model_idx'].tolist()
    fold_top_10_models = [initial_models[i] for i in fold_top_10_indices]
    print(f"  Selected top 10 models for tuning (recall range: {fold_screening_df.head(10)['mean_recall'].min():.4f} - {fold_screening_df.head(10)['mean_recall'].max():.4f})")
    
    # ========================================================================
    # STEP 2: TUNE TOP 10 MODELS
    # Store INNER CV scores for selection
    # ========================================================================
    print(f"  Tuning top 10 models using inner CV...")
    fold_candidates = []
    for model_idx, model in zip(fold_top_10_indices, fold_top_10_models):
        model_name = fold_screening_df[fold_screening_df['model_idx'] == model_idx]['model_name'].iloc[0]
        model_type = type(model).__name__
        
        if model_type in objective_map:
            # tune with Optuna using inner CV
            print(f"    Tuning {model_name}...", end=" ")
            objective_fn = objective_map[model_type](X_outer_train, y_outer_train, inner_cv, fold_features)
            study = optuna.create_study(direction="maximize")
            study.optimize(objective_fn, n_trials=50, show_progress_bar=False)
            
            inner_cv_score = study.best_value
            
            # build best model (for later evaluation if selected)
            best_params = study.best_params
            if model_type == 'LogisticRegression':
                best_model = LogisticRegression(**best_params, max_iter=2000, random_state=RANDOM_STATE)
            elif model_type == 'RandomForestClassifier':
                best_model = RandomForestClassifier(**best_params, random_state=RANDOM_STATE, n_jobs=-1)
            elif model_type == 'SVC':
                best_model = SVC(**best_params, probability=True, random_state=RANDOM_STATE)
            elif model_type == 'KNeighborsClassifier':
                best_model = KNeighborsClassifier(**best_params)
            else:
                best_model = model
            
            print(f"inner CV score: {inner_cv_score:.4f}")
        else:
            # use base model, use screening score as inner CV score
            inner_cv_score = fold_screening_df[fold_screening_df['model_idx'] == model_idx]['mean_recall'].iloc[0]
            best_model = model
            print(f"    {model_name} (base model): inner CV score: {inner_cv_score:.4f}")
        
        fold_candidates.append({
            'model_idx': model_idx,
            'model_name': model_name,
            'model_type': model_type,
            'model': best_model,
            'inner_cv_score': inner_cv_score, 
        })
    
    # track best candidate per model family for ensemble selection
    for candidate in fold_candidates:
        ftype = candidate['model_type']
        if ftype not in family_candidates:
            family_candidates[ftype] = []
        family_candidates[ftype].append({
            'fold': outer_fold,
            'model': candidate['model'],
            'inner_cv_score': candidate['inner_cv_score'],
            'model_name': candidate['model_name'],
            'model_idx': candidate['model_idx'],
        })
    
    # ========================================================================
    # STEP 3: SELECT WINNER BASED ON INNER CV ONLY 
    # ========================================================================
    fold_winner = max(fold_candidates, key=lambda x: x['inner_cv_score'])
    print(f"\n  → Fold {outer_fold} selected: {fold_winner['model_name']} (inner CV: {fold_winner['inner_cv_score']:.4f})")
    
    # ========================================================================
    # STEP 4: EVALUATE THE ONE SELECTED MODEL ON outer_val
    # ========================================================================
    print(f"     Evaluating on outer_val...", end=" ")
    pipe = make_pipeline(fold_winner['model'], fold_features)
    pipe.fit(X_outer_train, y_outer_train)
    y_pred = pipe.predict(X_outer_val)
    
    # outer_val score is used for evaluation only
    accuracy_val = accuracy_score(y_outer_val, y_pred)
    precision_val = precision_score(y_outer_val, y_pred, average='macro', zero_division=0)
    recall_val = recall_score(y_outer_val, y_pred, average='macro', zero_division=0)
    f1_val = f1_score(y_outer_val, y_pred, average='macro', zero_division=0)
    
    print(f"outer_val recall: {recall_val:.4f}")
    
    # store winner's score (one per fold, unbiased estimate)
    outer_selected_scores.append({
        'fold': outer_fold,
        'model_idx': fold_winner['model_idx'],
        'model_name': fold_winner['model_name'],
        'model_type': fold_winner['model_type'],
        'inner_cv_score': fold_winner['inner_cv_score'],
        'accuracy': accuracy_val,
        'precision': precision_val,
        'recall': recall_val,
        'f1': f1_val,
    })
    
    # track win counts for diagnostics
    if fold_winner['model_idx'] not in model_win_counts:
        model_win_counts[fold_winner['model_idx']] = 0
    model_win_counts[fold_winner['model_idx']] += 1

# ============================================================================
# AGGREGATE RESULTS ACROSS OUTER FOLDS
# ============================================================================
print("\n" + "=" * 80)
print("NESTED CV RESULTS")
print("=" * 80)
print("\n NESTED CV:")
print("   • Models selected using ONLY inner CV scores")
print("   • outer_val used ONLY for evaluation (never for selection)")
print("   • Each fold contributes exactly one unbiased score")
print("   • This is a valid estimate of the full model selection procedure\n")

if len(outer_selected_scores) > 0:
    selected_df = pd.DataFrame(outer_selected_scores)
    print("=" * 80)
    print("NESTED CV ESTIMATE (Best Single Model Per Fold)")
    print("NOTE: scores reflect single-model selection per fold, not the")
    print("      VotingClassifier ensemble. Ensemble CV would require re-running")
    print("      the full ensemble in each outer fold (computationally expensive).")
    print("=" * 80)
    print(f"  Accuracy:  {selected_df['accuracy'].mean():.4f} ± {selected_df['accuracy'].std():.4f}")
    print(f"  Precision: {selected_df['precision'].mean():.4f} ± {selected_df['precision'].std():.4f}")
    print(f"  Recall:    {selected_df['recall'].mean():.4f} ± {selected_df['recall'].std():.4f}")
    print(f"  F1:        {selected_df['f1'].mean():.4f} ± {selected_df['f1'].std():.4f}")
    print("=" * 80)
    
    # diagnostics per model: how many folds each model won
    print("\n" + "=" * 80)
    print("PER-MODEL WIN COUNTS (Diagnostics)")
    print("=" * 80)
    win_summary = []
    for model_idx, win_count in sorted(model_win_counts.items(), key=lambda x: x[1], reverse=True):
        # get model name from selected scores
        model_scores = [s for s in outer_selected_scores if s['model_idx'] == model_idx]
        if model_scores:
            model_name = model_scores[0]['model_name']
            avg_inner_cv = np.mean([s['inner_cv_score'] for s in model_scores])
            avg_recall = np.mean([s['recall'] for s in model_scores])
            win_summary.append({
                'model_idx': model_idx,
                'model_name': model_name,
                'wins': win_count,
                'avg_inner_cv_score': avg_inner_cv,
                'avg_outer_val_recall': avg_recall,
            })
    
    win_df = pd.DataFrame(win_summary)
    if len(win_df) > 0:
        print(win_df.to_string(index=False))
    print("=" * 80)
    
    # ========================================================================
    # SELECT TOP-3 FAMILIES FOR ENSEMBLE
    # ========================================================================
    family_best = {}
    for ftype, candidates in family_candidates.items():
        best = max(candidates, key=lambda x: x['inner_cv_score'])
        family_best[ftype] = best

    sorted_families = sorted(family_best.items(), key=lambda x: x[1]['inner_cv_score'], reverse=True)
    top3_families = sorted_families[:3]

    print("\n" + "=" * 80)
    print("TOP-3 FAMILIES SELECTED FOR ENSEMBLE (by best inner CV recall_macro)")
    print("=" * 80)
    for rank, (ftype, info) in enumerate(top3_families, 1):
        print(f"  {rank}. {ftype}: {info['model_name']} (inner CV: {info['inner_cv_score']:.4f})")
    print("=" * 80)

    # ========================================================================
    # PREPROCESS FULL TRAINING SET
    # ========================================================================
    X_train_final = X_train_raw.copy()
    y_train_final = y_train.copy()

    # step 1: drop columns with >NAN_COL_THRESHOLD NaN
    nan_pct_final = X_train_final.isna().mean()
    cols_to_drop_final = nan_pct_final[nan_pct_final > NAN_COL_THRESHOLD].index.tolist()
    if cols_to_drop_final:
        X_train_final = X_train_final.drop(columns=cols_to_drop_final)

    # step 2: drop rows with remaining NaNs
    mask_train = ~X_train_final.isna().any(axis=1)
    X_train_final = X_train_final[mask_train]
    y_train_final = y_train_final[mask_train]

    final_features = list(X_train_final.columns)
    X_train_final = X_train_final[final_features]

    print(f"\nPreprocessing full training set for ensemble model:")
    print(f"  Dropped {len(cols_to_drop_final)} columns with >{int(NAN_COL_THRESHOLD*100)}% NaN: {cols_to_drop_final if len(cols_to_drop_final) <= 5 else cols_to_drop_final[:5] + ['...']}")
    print(f"  Dropped {len(y_train) - len(y_train_final)} rows with remaining NaNs")
    print(f"  Final training shape: {X_train_final.shape}")

    # ========================================================================
    # RETUNE EACH OF THE 3 SELECTED MODELS ON FULL TRAINING SET
    # ========================================================================
    from sklearn.ensemble import VotingClassifier

    ensemble_estimators = []

    for ftype, info in top3_families:
        model_idx = info['model_idx']
        base_model = initial_models[model_idx]
        print(f"\nProcessing {ftype} ({info['model_name']})...")

        if ftype in objective_map:
            print(f"  Retuning on full training set (150 Optuna trials)...")
            objective_fn = objective_map[ftype](X_train_final, y_train_final, inner_cv, final_features)
            study = optuna.create_study(direction="maximize")
            study.optimize(objective_fn, n_trials=150, show_progress_bar=False)
            best_params = study.best_params
            if ftype == 'LogisticRegression':
                tuned_clf = LogisticRegression(**best_params, max_iter=2000, random_state=RANDOM_STATE)
            elif ftype == 'RandomForestClassifier':
                tuned_clf = RandomForestClassifier(**best_params, random_state=RANDOM_STATE, n_jobs=-1)
            elif ftype == 'SVC':
                tuned_clf = SVC(**best_params, probability=True, random_state=RANDOM_STATE)
            elif ftype == 'KNeighborsClassifier':
                tuned_clf = KNeighborsClassifier(**best_params)
            else:
                tuned_clf = base_model  # fallback: unknown type, use base model
            print(f"  Best retuning score: {study.best_value:.4f} | params: {best_params}")
        else:
            tuned_clf = base_model  # type not in objective_map: use base model as-is

        # short tag for VotingClassifier estimator name
        short_name = {
            'LogisticRegression': 'lr',
            'RandomForestClassifier': 'rf',
            'SVC': 'svc',
            'KNeighborsClassifier': 'knn',
        }.get(ftype, ftype.lower()[:4])
        ensemble_estimators.append((short_name, tuned_clf))

    # ========================================================================
    # BUILD SOFT VOTING CLASSIFIER
    # ========================================================================
    voting_clf = VotingClassifier(estimators=ensemble_estimators, voting='soft')
    best_model = Pipeline([
        ("uncertainty", UncertaintyTransformer(
            feature_names=final_features,
            class_labels=None,
            n_bins=N_BINS,
            eps=EPS,
        )),
        ("scaler", StandardScaler()),
        ("clf", voting_clf),
    ])
    best_model.fit(X_train_final, y_train_final)

    family_names = [ft for ft, _ in top3_families]
    best_model_name = "VotingClassifier(" + "+".join(family_names) + ")"
    print(f"\n✓ Ensemble built and fitted: {best_model_name}")

    tuned_models = {best_model_name: best_model}

    # CV results: aggregate all outer fold scores as unbiased ensemble estimate
    if outer_selected_scores:
        tuned_results = {
            best_model_name: {
                'n_folds': len(outer_selected_scores),
                'accuracy_mean': np.mean([s['accuracy'] for s in outer_selected_scores]),
                'accuracy_std': np.std([s['accuracy'] for s in outer_selected_scores]),
                'precision_mean': np.mean([s['precision'] for s in outer_selected_scores]),
                'precision_std': np.std([s['precision'] for s in outer_selected_scores]),
                'recall_mean': np.mean([s['recall'] for s in outer_selected_scores]),
                'recall_std': np.std([s['recall'] for s in outer_selected_scores]),
                'f1_mean': np.mean([s['f1'] for s in outer_selected_scores]),
                'f1_std': np.std([s['f1'] for s in outer_selected_scores]),
            }
        }
    else:
        tuned_results = {best_model_name: {}}
else:
    print("  No outer fold results found!")
    top3_families = []
    tuned_models = {}
    tuned_results = {}
    best_model_name = None
    final_features = []


NESTED CROSS-VALIDATION
Outer CV: 5-fold StratifiedKFold (evaluation)
Inner CV: 5-fold StratifiedKFold × 5 repeats (tuning)
Screening CV: 5-fold StratifiedKFold (model screening)

Evaluating all 20 models, selecting top 10 per outer fold...


OUTER FOLD 1/5
  Preprocessing (fold 1):
    Dropped 1 columns with >20% NaN: ['ALBUMIN']
    Dropped 14 rows with remaining NaNs from outer_train
    Final outer_train shape: (103, 45)
    Final outer_val shape: (25, 45)
  Screening all 20 models on outer_train...


[I 2026-03-18 23:16:33,372] A new study created in memory with name: no-name-e7fce503-62d3-4a85-8d98-9ddfb8d8364e
[I 2026-03-18 23:16:33,375] Trial 0 pruned. 
[I 2026-03-18 23:16:33,375] Trial 1 pruned. 


  Selected top 10 models for tuning (recall range: 0.9125 - 0.9625)
  Tuning top 10 models using inner CV...
    Tuning LogisticRegression_5... 

[I 2026-03-18 23:16:34,013] Trial 2 finished with value: 0.8902136752136751 and parameters: {'penalty': 'l2', 'solver': 'liblinear', 'class_weight': 'balanced', 'C': 0.0009116407501816494}. Best is trial 2 with value: 0.8902136752136751.
[I 2026-03-18 23:16:34,624] Trial 3 finished with value: 0.8072008547008548 and parameters: {'penalty': 'l1', 'solver': 'liblinear', 'class_weight': None, 'C': 0.05885594971866559}. Best is trial 2 with value: 0.8902136752136751.
[I 2026-03-18 23:16:35,296] Trial 4 finished with value: 0.5 and parameters: {'penalty': 'l2', 'solver': 'saga', 'class_weight': None, 'C': 6.11552698266588e-05}. Best is trial 2 with value: 0.8902136752136751.
[I 2026-03-18 23:16:35,297] Trial 5 pruned. 
[I 2026-03-18 23:16:35,297] Trial 6 pruned. 
[I 2026-03-18 23:16:35,298] Trial 7 pruned. 
[I 2026-03-18 23:16:35,919] Trial 8 finished with value: 0.9152777777777777 and parameters: {'penalty': 'l2', 'solver': 'lbfgs', 'class_weight': 'balanced', 'C': 0.0004099247803237165}. 

inner CV score: 0.9580
    Tuning RandomForestClassifier_6... 

...

  Best retuning score: 0.9629 | params: {'penalty': 'l2', 'solver': 'saga', 'class_weight': None, 'C': 1.25008864044676}

Processing SVC (SVC_13)...
  Retuning on full training set (150 Optuna trials)...


[I 2026-03-19 01:57:45,356] Trial 0 finished with value: 0.8824722222222223 and parameters: {'kernel': 'rbf', 'C': 1.350825001630544, 'gamma': 'auto'}. Best is trial 0 with value: 0.8824722222222223.
[I 2026-03-19 01:57:45,868] Trial 1 finished with value: 0.8786666666666667 and parameters: {'kernel': 'sigmoid', 'C': 0.21986171173342164, 'gamma': 'auto'}. Best is trial 0 with value: 0.8824722222222223.
[I 2026-03-19 01:57:46,378] Trial 2 finished with value: 0.919611111111111 and parameters: {'kernel': 'linear', 'C': 0.006707976965971759}. Best is trial 2 with value: 0.919611111111111.
[I 2026-03-19 01:57:46,872] Trial 3 finished with value: 0.5 and parameters: {'kernel': 'rbf', 'C': 0.052514607041562404, 'gamma': 'scale'}. Best is trial 2 with value: 0.919611111111111.
[I 2026-03-19 01:57:47,408] Trial 4 finished with value: 0.5 and parameters: {'kernel': 'linear', 'C': 0.0011393381276567252}. Best is trial 2 with value: 0.919611111111111.
[I 2026-03-19 01:57:47,940] Trial 5 finished 

  Best retuning score: 0.9584 | params: {'kernel': 'linear', 'C': 0.013956906753566844}

Processing RandomForestClassifier (RandomForestClassifier_8)...
  Retuning on full training set (150 Optuna trials)...


[I 2026-03-19 01:59:05,250] Trial 0 finished with value: 0.8982777777777778 and parameters: {'n_estimators': 450, 'max_depth': 19, 'min_samples_split': 18, 'min_samples_leaf': 4, 'max_features': None, 'class_weight': None}. Best is trial 0 with value: 0.8982777777777778.
[I 2026-03-19 01:59:06,225] Trial 1 finished with value: 0.9170277777777778 and parameters: {'n_estimators': 100, 'max_depth': 33, 'min_samples_split': 12, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'class_weight': 'balanced'}. Best is trial 1 with value: 0.9170277777777778.
[I 2026-03-19 01:59:07,178] Trial 2 finished with value: 0.9157500000000001 and parameters: {'n_estimators': 100, 'max_depth': 40, 'min_samples_split': 12, 'min_samples_leaf': 3, 'max_features': 'log2', 'class_weight': 'balanced'}. Best is trial 1 with value: 0.9170277777777778.
[I 2026-03-19 01:59:08,119] Trial 3 finished with value: 0.8837222222222222 and parameters: {'n_estimators': 100, 'max_depth': 36, 'min_samples_split': 4, 'min_samples_

  Best retuning score: 0.9338 | params: {'n_estimators': 650, 'max_depth': 33, 'min_samples_split': 12, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'class_weight': 'balanced'}

✓ Ensemble built and fitted: VotingClassifier(LogisticRegression+SVC+RandomForestClassifier)


In [10]:
# ============================================================================
# FINAL TEST SET EVALUATION
# ============================================================================

RUN_FINAL_TEST = True  # set to True only for final evaluation

if not RUN_FINAL_TEST:
    print("\n" + "="*80)
    print("  FINAL TEST EVALUATION IS DISABLED")
    print("="*80)
else:
    print("\n" + "="*80)
    print(" RUNNING FINAL TEST EVALUATION")
    print("="*80)
    
    # NOW materialize the test data
    df_test = df.loc[test_idx].copy()
    
    # ========================================================================
    # PREPROCESSING FOR TEST SET
    # ========================================================================
    
    # derive features directly from the fitted model to guarantee consistency
    # (avoids any divergence from recomputing column-dropping logic independently)
    final_features = list(best_model.named_steps['uncertainty'].feature_names_in_)
    
    # extract test set using only the final features
    X_test_raw = df_test[final_features].copy()
    y_test = df_test[LABEL_COL].values
    
    # drop rows with NaNs from test set (complete-case analysis)
    initial_test = len(X_test_raw)
    mask_test = ~X_test_raw.isna().any(axis=1)
    X_test_raw = X_test_raw[mask_test]
    y_test = y_test[mask_test]
    dropped_test = initial_test - len(X_test_raw)
    
    
    print(f"\nTest set preprocessing:")
    print(f"  Initial test samples: {initial_test}")
    print(f"  Dropped (incomplete records): {dropped_test}")
    print(f"  Final test samples: {len(X_test_raw)}")
    print(f"  Using {len(final_features)} features (after column dropping)")
    
    # verify no NaNs
    assert X_test_raw.isna().sum().sum() == 0, "Test set still has NaNs!"
    print(f"  ✓ Verified: Zero NaN values in test set")
    print(f"  Using {len(final_features)} features (same as final model training)")
    print(f"  Test set shape: {X_test_raw.shape}")
    
    print("="*80)
    
    # get the best model 
    best_model_name = list(tuned_models.keys())[0]
    best_model = tuned_models[best_model_name]
    
    print(f"Evaluating best model: {best_model_name}")
    print(f"(Top-3 families by nested CV, soft-voting ensemble, retrained on full training set)")
    
    # evaluate on test set
    y_pred = best_model.predict(X_test_raw)
    
    # calculate metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_test, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
    
    print("\nFinal Model Performance on Test Set:")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  Precision (macro): {prec:.4f}")
    print(f"  Recall (macro): {rec:.4f}")
    print(f"  F1-score (macro): {f1:.4f}")
    
    cm = confusion_matrix(y_test, y_pred, labels=[1, 2])
    print("\n=== CONFUSION MATRIX (rows=true, cols=pred) ===")
    print("                Predicted")
    print("              Myocarditis  ACS")
    print(f"True Myocarditis    {cm[0,0]:4d}    {cm[0,1]:4d}")
    print(f"True ACS            {cm[1,0]:4d}    {cm[1,1]:4d}")
    
    # classification report
    print("\n=== CLASSIFICATION REPORT ===")
    print(classification_report(y_test, y_pred, labels=[1, 2], target_names=['Myocarditis', 'ACS']))
    
    final_metrics = {
        'model_name': best_model_name,
        'accuracy': float(acc),
        'precision_macro': float(prec),
        'recall_macro': float(rec),
        'f1_macro': float(f1),
        'confusion_matrix': cm.tolist()
    }
    
    with open('final_test_metrics.json', 'w') as f:
        import json
        json.dump(final_metrics, f, indent=2)
    
    print("\n" + "="*80)
    print(" FINAL TEST EVALUATION COMPLETE")
    print("="*80)
    print(f"Final metrics saved to: final_test_metrics.json")
    print("="*80)



 RUNNING FINAL TEST EVALUATION

Test set preprocessing:
  Initial test samples: 37
  Dropped (incomplete records): 5
  Final test samples: 32
  Using 45 features (after column dropping)
  ✓ Verified: Zero NaN values in test set
  Using 45 features (same as final model training)
  Test set shape: (32, 45)
Evaluating best model: VotingClassifier(LogisticRegression+SVC+RandomForestClassifier)
(Top-3 families by nested CV, soft-voting ensemble, retrained on full training set)

Final Model Performance on Test Set:
  Accuracy: 0.9375
  Precision (macro): 0.9500
  Recall (macro): 0.9286
  F1-score (macro): 0.9352

=== CONFUSION MATRIX (rows=true, cols=pred) ===
                Predicted
              Myocarditis  ACS
True Myocarditis      12       2
True ACS               0      18

=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

 Myocarditis       1.00      0.86      0.92        14
         ACS       0.90      1.00      0.95        18

    accuracy      

In [11]:
# SAVE MODEL AND TRANSFORMATION PARAMETERS, will be used in report
print("\n" + "=" * 80)
print("SAVING MODEL")
print("=" * 80)

# save best individual model, used for test evaluation
with open('best_model_finetuned.pkl', 'wb') as f:
    pickle.dump(best_model, f)

# extract class labels from fitted transformer
fitted_class_labels = best_model.named_steps['uncertainty'].classes_

# Get CV results for the best model
best_model_cv = tuned_results.get(best_model_name, {})

# IMPORTANT: Save final_features (what model actually uses), not original features
# final_features is defined after preprocessing in Cell 8 and matches the trained model
with open('model_metadata.pkl', 'wb') as f:
    pickle.dump({
        'features': final_features,  # Use final_features (after column dropping)
        'class_labels': fitted_class_labels,
        'n_bins': N_BINS,
        'eps': EPS,
        'model_name': best_model_name,
        'cv_results': {
            'accuracy_mean': best_model_cv.get('accuracy_mean', None),
            'accuracy_std': best_model_cv.get('accuracy_std', None),
            'precision_mean': best_model_cv.get('precision_mean', None),
            'precision_std': best_model_cv.get('precision_std', None),
            'recall_mean': best_model_cv.get('recall_mean', None),
            'recall_std': best_model_cv.get('recall_std', None),
            'f1_mean': best_model_cv.get('f1_mean', None),
            'f1_std': best_model_cv.get('f1_std', None),
            'n_folds_selected': best_model_cv.get('n_folds', best_model_cv.get('n_folds_selected', None)),
        }
    }, f)

print("Saved:")
print("  - best_model_finetuned.pkl")
print("  - model_metadata.pkl")
print(f"\nMetadata info:")
print(f"  Model: {best_model_name}")
print(f"  Features saved: {len(final_features)} (after preprocessing)")
print(f"  Original features: {len(features)}")
if len(final_features) < len(features):
    dropped = set(features) - set(final_features)
    print(f"  Dropped columns: {dropped}")
print(f"\nCV Results (nested CV, outer fold evaluation):")
if best_model_cv:
    print(f"  Accuracy:  {best_model_cv.get('accuracy_mean', 0):.4f} ± {best_model_cv.get('accuracy_std', 0):.4f}")
    print(f"  Precision: {best_model_cv.get('precision_mean', 0):.4f} ± {best_model_cv.get('precision_std', 0):.4f}")
    print(f"  Recall:    {best_model_cv.get('recall_mean', 0):.4f} ± {best_model_cv.get('recall_std', 0):.4f}")
    print(f"  F1:        {best_model_cv.get('f1_mean', 0):.4f} ± {best_model_cv.get('f1_std', 0):.4f}")
print("\n" + "=" * 80)


SAVING MODEL
Saved:
  - best_model_finetuned.pkl
  - model_metadata.pkl

Metadata info:
  Model: VotingClassifier(LogisticRegression+SVC+RandomForestClassifier)
  Features saved: 45 (after preprocessing)
  Original features: 46
  Dropped columns: {'ALBUMIN'}

CV Results (nested CV, outer fold evaluation):
  Accuracy:  0.9393 ± 0.0475
  Precision: 0.9428 ± 0.0441
  Recall:    0.9356 ± 0.0538
  F1:        0.9356 ± 0.0494



In [12]:
# ============================================================================
# SANITY CHECK
# ============================================================================

print("Running sanity check on first outer fold...\n")

# Get first fold
outer_cv_check = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
outer_train_idx, outer_val_idx = next(outer_cv_check.split(X_train_raw, y_train))

X_outer_train_check = X_train_raw.iloc[outer_train_idx].copy()
y_outer_train_check = y_train[outer_train_idx].copy()
X_outer_val_check = X_train_raw.iloc[outer_val_idx].copy()
y_outer_val_check = y_train[outer_val_idx].copy()

# Apply same preprocessing as in outer CV loop (columns first, then rows)
nan_pct_check = X_outer_train_check.isna().mean()
cols_to_drop_check = nan_pct_check[nan_pct_check > NAN_COL_THRESHOLD].index.tolist()

if cols_to_drop_check:
    X_outer_train_check = X_outer_train_check.drop(columns=cols_to_drop_check)
    X_outer_val_check = X_outer_val_check.drop(columns=cols_to_drop_check, errors='ignore')

mask_train = ~X_outer_train_check.isna().any(axis=1)
X_outer_train_check = X_outer_train_check[mask_train]
y_outer_train_check = y_outer_train_check[mask_train]

fold_features_check = list(X_outer_train_check.columns)
X_outer_val_check = X_outer_val_check[fold_features_check]

# drop rows with NaNs from validation set
mask_val = ~X_outer_val_check.isna().any(axis=1)
X_outer_val_check = X_outer_val_check[mask_val]
y_outer_val_check = y_outer_val_check[mask_val]

# Check 1: fold_features matches columns
assert fold_features_check == list(X_outer_train_check.columns), \
    f"❌ fold_features mismatch: {set(fold_features_check) ^ set(X_outer_train_check.columns)}"
print("✓ Check 1 passed: fold_features matches X_outer_train.columns")

# Check 2: Pipeline fits without KeyError
test_model = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
test_pipe = make_pipeline(test_model, fold_features_check)
try:
    test_pipe.fit(X_outer_train_check, y_outer_train_check)
    print("✓ Check 2 passed: Pipeline fits without KeyError")
except KeyError as e:
    print(f"❌ Check 2 failed: KeyError during fit: {e}")
    raise

# Check 3: Pipeline can predict
try:
    y_pred_check = test_pipe.predict(X_outer_val_check)
    assert len(y_pred_check) == len(y_outer_val_check), \
        f"❌ Prediction length mismatch: {len(y_pred_check)} vs {len(y_outer_val_check)}"
    print("✓ Check 3 passed: Pipeline can predict on X_outer_val")
except Exception as e:
    print(f"❌ Check 3 failed: Error during predict: {e}")
    raise

print(f"\n✅ All sanity checks passed!")
print(f"   fold_features length: {len(fold_features_check)}")
print(f"   X_outer_train shape: {X_outer_train_check.shape}")
print(f"   X_outer_val shape: {X_outer_val_check.shape}")

Running sanity check on first outer fold...

✓ Check 1 passed: fold_features matches X_outer_train.columns
✓ Check 2 passed: Pipeline fits without KeyError
✓ Check 3 passed: Pipeline can predict on X_outer_val

✅ All sanity checks passed!
   fold_features length: 45
   X_outer_train shape: (103, 45)
   X_outer_val shape: (25, 45)


In [13]:
# helper to load the model

def load_finetuned_model(pickle_path='best_model_finetuned.pkl'):
    """
    Safely load the finetuned model from pickle.
    
    This function checks that UncertaintyTransformer is imported before loading.
    If not, it raises a helpful error message.
    
    Args:
        pickle_path: Path to the pickle file (default: 'best_model_finetuned.pkl')
    
    Returns:
        The loaded model (Pipeline with UncertaintyTransformer)
    
    Raises:
        ImportError: If UncertaintyTransformer cannot be imported from uncertainty_transformer module
    """
    try:
        _ = UncertaintyTransformer
    except NameError:
        raise ImportError(
            "UncertaintyTransformer is not imported. "
            "Please run: from uncertainty_transformer import UncertaintyTransformer"
        )
    
    # load the model
    with open(pickle_path, 'rb') as f:
        model = pickle.load(f)
    
    print(f"✓ Model loaded successfully from {pickle_path}")
    return model